# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [9]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': ['id, name, email'],
    'definition': ['stores user account information like name and email']
}   
df = pd.DataFrame(data)
print(df)

             table                                         definition
0  id, name, email  stores user account information like name and ...


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [11]:
print(text_tables)

id, name, email: stores user account information like name and email


In [12]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [27]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="Show all users and their emails")

In [28]:
print(return_OAI(pqt1))

```json
{
    "tables": ["users"]
}
```


In [17]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question='Show all trips with their start and end dates')

In [18]:
print(return_OAI(pqt3))

```json
{
    "tables": ["trips"]
}
```


*Fitness App*

In [38]:
#Fitness App
data2 = {
    'table': ['users', 'workouts', 'exercises'],
    'definition': [
        'stores user profile info',
        'stores workout sessions with date and duration',
        'stores exercise details like name and muscle group'
    ]
}
df2 = pd.DataFrame(data2)
print(df2)

       table                                         definition
0      users                           stores user profile info
1   workouts     stores workout sessions with date and duration
2  exercises  stores exercise details like name and muscle g...


In [49]:
text_tables2 = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df2.iterrows()])
print(text_tables2)

users: stores user profile info
workouts: stores workout sessions with date and duration
exercises: stores exercise details like name and muscle group


In [65]:
prompt_question_tables2 = """
You are working with a fitness tracking database.

Below are the available tables and a short description of what each contains:

### Available Tables
{tables}

Your task:
Identify which tables are required to build a SQL query that answers the user's request.

Constraints:
- Only choose from the tables listed above
- Do not assume or create new tables
- Return your answer strictly as a JSON array of table names
- No explanations, no extra text

### User Request:
{question}
"""

In [66]:
pqt_1 = prompt_question_tables2.format(
    tables=text_tables2,
    question="Show all workouts longer than 60 minutes"
)

print(return_OAI(pqt_1))

```json
["workouts"]
```


In [67]:
pqt_2 = prompt_question_tables2.format(
    tables=text_tables2,
    question="Which exercises target the chest muscle?"
)

print(return_OAI(pqt_2))


```json
["exercises"]
```


In [68]:
pqt_3 = prompt_question_tables2.format(
    tables=text_tables2,
    question="List all users who completed at least 5 workouts"
)

print(return_OAI(pqt_3))


[
    "users",
    "workouts"
]


In [69]:
pqt_4 = prompt_question_tables2.format(
    tables=text_tables2,
    question="Show all exercises used in workouts during the last week"
)

print(return_OAI(pqt_4))


```json
["workouts", "exercises"]
```


*Library App*

In [70]:
data3 = {
    'table': ['books', 'authors', 'borrowings'],
    'definition': [
        'stores book info like title, genre and author_id',
        'stores author details like name and nationality',
        'stores records of which user borrowed which book and when'
    ]
}

df3 = pd.DataFrame(data3)
print(df3)

        table                                         definition
0       books   stores book info like title, genre and author_id
1     authors    stores author details like name and nationality
2  borrowings  stores records of which user borrowed which bo...


In [71]:
text_tables3 = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df3.iterrows()])
print(text_tables3)

books: stores book info like title, genre and author_id
authors: stores author details like name and nationality
borrowings: stores records of which user borrowed which book and when


In [72]:
prompt_question_tables3 = """
You are working with a library database.

Below are the available tables and a short description of what each contains:

### Available Tables
{tables}

Your task:
Identify which tables are required to build a SQL query that answers the user's request.

Constraints:
- Only choose from the tables listed above
- Do not assume or create new tables
- Return your answer strictly as a JSON array of table names
- No explanations, no extra text

### User Request:
{question}
"""

In [73]:
pqt_5 = prompt_question_tables3.format(
    tables=text_tables3,
    question="Show all books in the fantasy genre"
)

print(return_OAI(pqt_5))


```json
["books"]
```


In [74]:
pqt_5 = prompt_question_tables3.format(
    tables=text_tables3,
    question="Which authors have written more than 3 books?"
)

print(return_OAI(pqt_5))

[
    "books",
    "authors"
]


In [75]:
pqt_6 = prompt_question_tables3.format(
    tables=text_tables3,
    question="Show all books borrowed in the last 30 days"
)

print(return_OAI(pqt_6))

[
    "books",
    "borrowings"
]


In [76]:
pqt_7 = prompt_question_tables3.format(
    tables=text_tables3,
    question="Which books are currently borrowed and not returned?"
)

print(return_OAI(pqt_7))

[
    "books",
    "borrowings"
]


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

*Report*


The experiments with both the fitness app and the library app show that the approach generally works, but with clear limitations. The model is able to select relevant tables, especially when the question requires combining multiple tables (e.g., users and workouts, or books and borrowings). However, the results are not fully reliable. In several cases, the model chose only the most obvious table and ignored potential relationships, which would be necessary for a correct SQL query in a real scenario.

Another issue is inconsistency in the output format. Even though the prompt explicitly required a JSON array, the model sometimes returned the result wrapped in markdown formatting, which would cause problems in automated systems. Additionally, the model appears to rely heavily on keyword matching rather than true understanding of database structure, meaning that clearer and more detailed table definitions significantly improve the results.

Overall, the approach demonstrates that GPT can assist in identifying relevant tables, but it is not robust enough to be used without additional validation, stricter prompting, or post-processing.

*What did I learn?*

GPT can correctly identify relevant tables, especially when a query clearly requires multiple tables (joins).
The quality of the result strongly depends on how clear and detailed the table definitions are.
The model often relies on keyword matching rather than true understanding of database relationships.
Output formatting is inconsistent (sometimes valid JSON, sometimes markdown), so post-processing is needed.
GPT is useful as a helper, but not reliable enough to be used alone without validation or stricter control.